# 検索拡張生成（RAG）システム、LangChain、ChromaDB 入門

このノートブックでは、情報を検索して応答を作成し、特定の情報に基づいて LLM を効果的にグラウンディング（接地）する質問応答システムの構築について説明します。事前トレーニングされた LLM、あるいはおそらくファインチューニングされた LLM でさえも、特定の（場合によっては非公開の）データやトレーニング データセットに含まれていなかった情報を理解するシステムが必要な場合、（それ単体では）十分ではありません。

このラボでは、次のことを行います。
* 検索拡張（Retrieval Augmented）システムのさまざまなコンポーネントについて学ぶ
* シンプルな検索拡張生成（RAG）システムを構築する
* LangChain と ChromaDB を使用してプロセスを簡素化およびスケーリングする

### セットアップ

In [ ]:
import pandas as pd
import scipy
from google import genai
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.chains import RetrievalQA
import wikipediaapi
from langchain_core.documents import Document
from langchain_classic.memory import ConversationBufferMemory
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
EMBEDDING_MODEL = "text-embedding-005"
GENERATIVE_MODEL = "gemini-3.8-flash"

### シンプルな検索拡張生成（RAG）システムの構築

このトイ（簡易）サンプルの例では、市販の LLM が知らないような情報に基づいて LLM をグラウンディングします。たとえば、2 匹のペットの世話をするハウスシッターのために残された指示などです。

In [ ]:
# LLM のグラウンディングに使用したい情報のリスト。
information = [
    "Estrella is a dog",
    "Finnegan is a cat",
    "Finnegan gets fed five times daily. Estrella gets fed three times daily.",
    "Estrella usually goes on one long walk per day, but needs to go outside every 4-6 hours",
    "Please play with Finnegan for 30 minutes each day. His favorite toy is the fake mouse!",
]

information_df = pd.DataFrame({"text": information})
information_df.head()


ほとんどの検索生成システムの中心にあるのはベクトル データベースです。ベクトル データベースは、情報の埋め込み表現を保存します。

テキストの埋め込み表現である列を情報の DataFrame に追加しましょう。[Text embeddings API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/embeddings/get-text-embeddings) を使用します。

In [ ]:
client = genai.Client(vertexai=True, location="global")

In [ ]:
information_df["vector"] = [
    x.values
    for x in client.models.embed_content(
        model=EMBEDDING_MODEL, contents=information
    ).embeddings
]
information_df.head()

検索システムには、指定されたクエリに回答するための最も関連性の高い情報を見つける方法が必要です。これは、最近傍（セマンティック類似度）検索によって行われます。クエリ（テキスト）入力を受け取り、情報内の各テキストに対する距離メトリクスを返す関数を定義しましょう。以下の処理を行う必要があります。
* 情報に使用したのと同じ埋め込みモデルでクエリを埋め込む
* クエリ ベクトルと各情報ベクトルとの間の距離メトリクスを計算する。使用可能な多くの類似度測定手法の 1 つであるコサイン類似度を使用します。
* クエリ ベクトルと各情報ベクトルとの間の距離メトリクスのリストを返す

In [ ]:
def embed_and_compute_distances(query: str):
    # クエリ文字列のベクトルを取得
    query_embedding = (
        client.models.embed_content(model=EMBEDDING_MODEL, contents=query)
        .embeddings[0]
        .values
    )  # クエリの埋め込み

    distances = []

    # クエリ ベクトルとすべての情報ベクトルとの間の距離を計算
    for _, row in information_df.iterrows():
        distances.append(
            {
                "information": row.text,
                "distance": scipy.spatial.distance.cosine(
                    query_embedding, row.vector
                ),
            }
        )

    return distances


この関数をサンプルでテストしてみましょう。

In [ ]:
embed_and_compute_distances(query="What type of animal is Estrella?")

"What type of animal is Estrella?"（Estrella はどんな種類の動物ですか？）のベクトルに最も近い（つまり最も類似している）ベクトルが、"Estrella is a dog"（Estrella は犬です）のベクトルであることに注目してください。これは、検索拡張システムを支える核心的な前提を浮き彫りにしています。それは、質問に答えるのに関連する情報は、ベクトル空間内で質問自体に近いということです。

あとは、最も近い情報ベクトルに対応するテキストをプロンプトに組み込み、そのプロンプトを LLM に送信してその情報をもとに質問に回答させる関数を作成するだけです。

まず、このプロンプトを作成するヘルパー関数の作成から始めます。`context` は関連する情報文字列（最近傍検索によって見つかったもの）になり、`query` はクエリ文字列になります。

In [ ]:
def get_prompt(query: str, context: list[str]):
    prompt = f"""
    Using only the provided context, answer the question.
    
    Context:
    {','.join(context)}
    
    Question: {query}.
    
    If you cannot answer the question using only the provided context, respond that you do not have the context needed to answer the question.
    """
    return prompt

次に、以下を行う関数にすべてをまとめます。
* クエリを埋め込む
* クエリ ベクトルとすべての情報ベクトルとの間の距離を計算する
* 距離でソートして、最も関連性の高い上位 k 個の情報テキストを取得する
* 最も関連性の高い k 個の情報テキストを、クエリとともに LLM へのプロンプトで使用する
* LLM のレスポンスと使用された情報（引用）を返す

In [ ]:
def retrieval_chain(query: str, k: int = 2):
    # クエリとすべての情報ベクトルとの間の距離を計算
    distances = embed_and_compute_distances(query)

    # 距離が小さい順（最も近い順）に情報をソート
    sorted_distances = sorted(distances, key=lambda x: x["distance"])

    # 最も近い k 個のベクトルに対応するテキストを取得
    closest_information_texts = [x["information"] for x in sorted_distances[:k]]

    # 最も近い k 個の情報テキストを LLM へのプロンプトに組み込む
    prompt = get_prompt(query, closest_information_texts)

    # プロンプトを LLM に送信
    response = client.models.generate_content(
        model=GENERATIVE_MODEL, contents=prompt
    )
    # レスポンスを返す

    print(f"Response: {response.text}")
    print(f"Information used: {closest_information_texts}")


In [ ]:
retrieval_chain("What type of animal is Estrella?")

In [ ]:
retrieval_chain("How many times a day do I need to feed Finnegan?")

In [ ]:
retrieval_chain("What stock should I invest in this month?")

プロンプトは、提供された情報からは答えられない質問がされた場合、LLM がそれに無理に答えようとしないように構築されていることに注目してください。

また、ここでは任意に k=2（プロンプトに最も近い 2 つの情報テキストを含める）を設定していることにも注意が必要です。ユースケースによって必要な k は異なり、万能な設定はありません。

### LangChain と Chroma による簡素化とスケーリング
もちろん、グラウンディング情報の例が 5 つしかない場合は、5 つすべてを簡単にプロンプトに含めることができます。言い換えれば、プロンプトに*何*が必要かを特定するための追加の検索ステップは不要でした。しかし、現実世界では数千または数百万のグラウンディング情報の例が存在する可能性があります。さらに、グラウンディングの例の数が増えるにつれて、すべての単一ベクトルに対して距離を計算するだけでは非常に非効率になります。つまり、本番環境の検索拡張生成システムには以下が必要です。
* 大量の情報を保存するためのスケーラブルなベクトル データベース
* 最近傍探索を実行するための効率的な方法

もちろん、[Vector Search](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/overview) のようなマネージドでスケーラブルな製品を含め、ベクトル ストアには多くの選択肢があります。わかりやすくするために、このラボではベクトル ストアとして [Chroma](https://python.langchain.com/docs/integrations/vectorstores/chroma) を使用し、検索システムをオーケストレーションするために [LangChain](https://github.com/langchain-ai/langchain) を使用します。LangChain は、上記のトイ サンプルで独自に実装する必要があった手順を簡素化するのに役立つクラスとメソッドを提供します。

#### ドキュメントの読み込み

このノートブックでは、Wikipedia API を使用して、大規模言語モデルに関する Wikipedia の記事のプライベート ナレッジベースを作成します。

In [ ]:
def search_wikipedia(query, language='en', limit=5):
    wiki_wiki = wikipediaapi.Wikipedia(user_agent='https://cloud.google.com/asl', language=language)
    docs = []
    results = wiki_wiki.search(query, limit=limit)
    for title, page in results.pages.items():
        doc = Document(
            page_content=page.text,
            metadata={
                'title': title,
                'source': page.canonicalurl
            }
        )
        docs.append(doc)
    return docs

docs = search_wikipedia("Large Language Models")

# 単一のドキュメントを確認
docs[0]


#### テキストをチャンクに分割
ドキュメントが準備できたので、チャンクに分割します。各チャンクがベクトル ストア内の 1 つのベクトルになります。これを行うには、チャンク サイズ（文字数）とチャンク オーバーラップ（重複量、つまりスライディング ウィンドウ）を定義します。最適なチャンク サイズを決定するのは難しい場合があります。チャンク サイズが大きすぎると、チャンクあたりの情報が多すぎることになり（個々のチャンクが十分に特定的でなくなる）、チャンク サイズが小さすぎると、チャンクあたりの情報が不十分になります。どちらの場合も、クエリ / 質問の埋め込みによる最近傍検索で実際に関連するチャンクを取得するのに苦労したり、チャンクが大きすぎて LLM クエリのコンテキストとして使用できなかったりして完全に失敗する可能性があります。

このノートブックでは、800 文字のチャンク サイズと 400 文字のチャンク オーバーラップを使用しますが、他のサイズも自由に試してみてください！注: Python の len 関数以外でチャンク サイズ / オーバーラップを決定したい場合は、`RecursiveCharacterTextSplitter` でカスタムの `length_function` を指定できます。`RecursiveCharacterTextSplitter` に加えて、検討できる[他のテキスト スプリッター](https://python.langchain.com/docs/how_to/#text-splitters) もあります。

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=400,
    length_function=len,
)

chunks = text_splitter.split_documents(docs)

# 最初の 2 つのチャンクを確認
chunks[0:2]


In [ ]:
print(f"Number of documents: {len(docs)}")
print(f"Number of chunks: {len(chunks)}")

#### ドキュメント チャンクの埋め込み
次に、ドキュメント チャンクを埋め込んでベクトル ストアに保存する必要があります。これには任意のテキスト埋め込みモデルを使用できますが、予測時にクエリ / 質問を埋め込むときにも同じテキスト埋め込みモデルを使用する必要があります。シンプルにするために、埋め込みには Gemini API を使用します。LangChain ライブラリは、Gemini Embeddings API の優れたラッパー クラスである `GoogleGenerativeAIEmbeddings` を提供しています。

Vector Search はインデックスの作成に時間がかかる（約 45 分）ため、シンプルにするために代わりに Chroma を使用します。もちろん、大規模なプライベート ナレッジベースを扱う実際のユースケースでは、すべてをメモリに収めることができない場合があります。LangChain には Chroma 用の使いやすいラッパー クラスがあり、ドキュメントのリストと埋め込みクラスを渡してベクトル ストアを作成できます。

In [ ]:
embedding = GoogleGenerativeAIEmbeddings(vertexai=True, model=EMBEDDING_MODEL)

# ベクトル ストアがディスクに保存されるように永続ディレクトリを設定
db = Chroma.from_documents(chunks, embedding, persist_directory="./vectorstore")


#### すべてを組み合わせる

すべてが整ったので、LangChain のチェーンでこれらすべてを結合します。LangChain のチェーンは、特定のユースケースで LLM を使用するために必要な複数のステップをオーケストレーションするだけです。この場合、連結するプロセスは、まずクエリ / 質問を埋め込み、次に最近傍検索を実行して関連するチャンクを見つけ、次に関連するチャンクを使用して LLM で応答を作成します。ベクトル ストアとして Chroma データベースを使用し、LLM として Gemini を使用します。LangChain は Gemini のラッパーである `GoogleGenerativeAI` を提供しています。

このシンプルな質問応答（Q/A）のユースケースでは、LangChain の `RetrievalQA` を使用してプロセスをリンクできます。

In [ ]:
# ベクトル ストア
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10},  # 取得する最近傍の数
)

# temperature、top_p、top_k を設定することもできます
llm = GoogleGenerativeAI(model=GENERATIVE_MODEL, max_output_tokens=1024, vertexai=True)

# 質問応答（Q/A）チェーン
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)


すべてが結合されたので、クエリを送信して回答を得ることができます！

In [ ]:
def ask_question(question: str):
    response = qa.invoke({"query": question})
    print(f"Response: {response['result']}\n")

    citations = {doc.metadata["source"] for doc in response["source_documents"]}
    print(f"Citations: {citations}\n")

    # 使用されたソース チャンクを出力するには下のコメントを解除
    # print(f"Source Chunks Used: {response['source_documents']}")


In [ ]:
ask_question("What technology underpins large language models?")

In [ ]:
ask_question("When was the transformer introduced?")

おめでとうございます！これで、検索拡張生成（RAG）システムをゼロから構築し、その知識を応用して、ベクトル データベースと LangChain によるオーケストレーションを使用した、より実践的なシステムを構築できました。